# 📈 Sistema GEBRA Portfolio v12 – Anatomia da Análise Grafotária (REVISADO)
**100% alinhado com o curso. Nenhuma invenção.**

In [ ]:
import os
EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')

# ==================== PARÂMETROS DO SISTEMA ====================
CAPITAL_TOTAL = 100000.0
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.10
PRECO_MINIMO = 5.00
MAX_SETUPS_POR_DIA = 5
MAX_ATIVOS_POR_SETOR = 2
PAYOFF_MINIMO = 3.0
ADX_MINIMO = 25
EFICIENCIA_MINIMA_CANDLE = 0.6
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
FALLBACK_TICKERS = ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'SUZB3', 'GGBR4', 'MGLU3', 'VVAR3', 'RENT3', 'RAIL3', 'CCRO3', 'ELET3', 'CPFE3', 'SBSP3', 'SANB11', 'B3SA3', 'JBSS3', 'BRFS3', 'KLBN11', 'EQTL3']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA', 'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']
VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
HABILITAR_LOGGING = True
ARQUIVO_LOG = "trading_log_v85.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v85.log"
MAX_DIAS_LOG = 30
MAX_PERDAS_CONSECUTIVAS = 3
DRAWDOWN_MAX_DIARIO = 0.02
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0
KELLY_FRAC = 0.30

# ==================== MÓDULOS DE ANÁLISE ====================
USAR_WYCKOFF = True
WYCKOFF_SENSIBILIDADE = 80
USAR_FIBONACCI = True
USAR_OBV = True
USAR_RSI = True
USAR_VOLUME = True
USAR_CANDLE = True

# ==================== SETUPS ATIVOS ====================
USAR_PIVO_ALTA = True
USAR_PIVO_BAIXA = True
USAR_FUNDO_DUPLO = True
USAR_TOPO_DUPLO = True
USAR_TOPO_TRIPLO = True
USAR_FUNDO_TRIPLO = True
USAR_TRIANGULO = True
USAR_RETANGULO_SETUP = True
USAR_BANDEIRA = True
USAR_FLAMULA = True
USAR_OCO = True
USAR_OCO_INVERTIDO = True
USAR_3_SOLDADOS = True
USAR_3_CORVOS = True
USAR_DIAMANTE = True
USAR_MARTELO_SUPORTE = True
USAR_ESTRELA_CADENTE = True
USAR_XICARA_ALCA = False
USAR_TOPO_FUNDO_ARREDONDADO = False
USAR_PULLBACK = False

# ==================== FERRAMENTAS OPCIONAIS ====================
USAR_ELLIOTT = False
USAR_ESTOCASTICO = False
USAR_MACD = False
USAR_BOLLINGER = False

total_setups = sum([USAR_PIVO_ALTA,USAR_PIVO_BAIXA,USAR_FUNDO_DUPLO,USAR_TOPO_DUPLO,USAR_TOPO_TRIPLO,USAR_FUNDO_TRIPLO,USAR_TRIANGULO,USAR_RETANGULO_SETUP,USAR_BANDEIRA,USAR_FLAMULA,USAR_OCO,USAR_OCO_INVERTIDO,USAR_3_SOLDADOS,USAR_3_CORVOS,USAR_DIAMANTE,USAR_MARTELO_SUPORTE,USAR_ESTRELA_CADENTE])
print("✅ Parâmetros v12 carregados")
print(f"   ADX: {ADX_MINIMO} | Payoff: {PAYOFF_MINIMO}:1 | Setups ativos: {total_setups}")

In [ ]:
!pip install yfinance pandas-ta requests-cache requests-ratelimiter gspread oauth2client python-dotenv --quiet 2>/dev/null
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, os, sys, traceback, gc, subprocess
from typing import Optional, Tuple, Dict, List, Any
from collections import Counter
from requests_cache import CachedSession
from requests_ratelimiter import LimiterSession
import gspread
from oauth2client.service_account import ServiceAccountCredentials
warnings.filterwarnings("ignore", category=FutureWarning)
try:
    from dotenv import load_dotenv
    load_dotenv()
    if not EMAIL_REMETENTE: EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
    if not SENHA_APP: SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')
except ImportError: pass
try:
    session = LimiterSession(per_second=0.4, session_factory=lambda: CachedSession(cache_name='yfinance.cache', backend='sqlite', expire_after=timedelta(hours=6)))
    yf.shared._requests = session
    print("✅ Cache YFinance ativado")
except Exception as e: print(f"⚠️ Cache: {e}")

class Logger:
    def __init__(self, al, ad=None):
        self.al = al; self.ad = ad; self.t0 = time.time()
        self.tm = {}; self.ct = {}; self.bf = []; self.mb = 100
    def log(self, m, n="INFO", t=None):
        ts = datetime.now().strftime("%H:%M:%S")
        msg = f"[{ts}] [{n}] {m}"
        if t: msg += f" | {t}"
        print(msg)
        if self.ad and HABILITAR_LOGGING:
            self.bf.append(msg+"\n")
            if len(self.bf) >= self.mb: self._fb()
    def _fb(self):
        if self.ad and self.bf:
            try:
                with open(self.ad,'a',encoding='utf-8') as f: f.writelines(self.bf)
                self.bf.clear()
            except Exception as e: print(f"Erro log: {e}")
    def warn(self,m,t=None): self.log(m,"WARN",t)
    def error(self,m,t=None): self.log(m,"ERRO",t)
    def inicio(self,n): self.tm[n]={'ini':time.time()}; self.log(f"🚀 {n}","ETAPA")
    def fim(self,n,d=None):
        if n in self.tm:
            dr = time.time()-self.tm[n]['ini']; self.tm[n]['dur']=dr
            msg = f"✅ {n} ({dr:.1f}s)"
            if d: msg += " | "+" | ".join(f"{k}:{v}" for k,v in d.items())
            self.log(msg,"ETAPA")
    def resumo(self):
        self._fb(); tt = time.time()-self.t0
        self.log("\n"+"="*60,"RESUMO"); self.log(f"⏱️ Total: {tt:.1f}s","RESUMO")
        for e,d in self.tm.items():
            if 'dur' in d: self.log(f"   • {e}: {d['dur']:.1f}s ({d['dur']/tt*100:.0f}%)","RESUMO")
        self.log("="*60+"\n","RESUMO")

logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)

def _log_exc(c, e):
    try:
        msg = f"[{c}] {type(e).__name__}: {str(e)[:200]}"
        if 'logger' in globals(): logger.error(msg)
        else: print(f"[ERRO] {msg}")
        with open('traceback_errors.log','a',encoding='utf-8') as f:
            f.write(f"\n{'='*60}\n{datetime.now()}\n{c}\n{str(e)}\n{traceback.format_exc()}")
    except: pass

def conectar_sheets():
    try:
        from google.colab import userdata
        kc = userdata.get('GCP_SERVICE_ACCOUNT_KEY')
        if not kc: return None,None,None
        jk = json.loads(kc)
        sc = ['https://spreadsheets.google.com/feeds','https://www.googleapis.com/auth/drive']
        cr = ServiceAccountCredentials.from_json_keyfile_dict(jk,sc)
        cl = gspread.authorize(cr)
        pid = userdata.get('GOOGLE_SHEET_ID')
        pl = cl.open_by_key(pid)
        return pl.worksheet("CircuitBreaker"),pl.worksheet("Ledger"),pl.worksheet("Scanner")
    except Exception as e:
        logger.warn(f"Sheets: {e}")
        return None,None,None

aba_circuit, aba_ledger, aba_scanner = conectar_sheets()

try:
    from google.colab import userdata
    TELEGRAM_TOKEN = userdata.get('TELEGRAM_TOKEN')
    TELEGRAM_CHAT_ID = userdata.get('TELEGRAM_CHAT_ID')
except:
    TELEGRAM_TOKEN = os.getenv('TELEGRAM_TOKEN','')
    TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID','')

def enviar_telegram(m, pm='HTML'):
    if not TELEGRAM_TOKEN or not TELEGRAM_CHAT_ID: return
    try: requests.post(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage", data={'chat_id':TELEGRAM_CHAT_ID,'text':m,'parse_mode':pm}, timeout=10)
    except Exception as e: _log_exc('Telegram',e)

print("✅ Célula 1 carregada")

In [ ]:
# ================ CÉLULA 2: FUNÇÕES AUXILIARES (MANTIDAS) ================
def _safe_divide(a,b,d=np.nan):
    return d if b is None or b==0 or pd.isna(b) else a/b
def _safe_log(x,d=np.nan):
    return d if x is None or x<=0 or pd.isna(x) else np.log(x)
def calcular_eficiencia_candle(df):
    corpo = abs(df['Close']-df['Open'])
    ss = df['High']-df[['Close','Open']].max(axis=1)
    si = df[['Close','Open']].min(axis=1)-df['Low']
    rt = df['High']-df['Low']; rt = rt.replace(0,np.nan)
    ef = pd.Series(index=df.index,dtype=float)
    al = df['Close']>df['Open']; bx = df['Close']<df['Open']
    ef[al] = 1-(ss[al]/rt[al]); ef[bx] = 1-(si[bx]/rt[bx])
    return ef
def detectar_regime(df,j=20):
    dt = df.copy(); dt['ret'] = dt['Close'].pct_change(); dt['vol'] = dt['ret'].rolling(j).std()
    try:
        ax = ta.adx(dt['High'],dt['Low'],dt['Close'],length=14)
        if ax is not None and not ax.empty:
            if isinstance(ax,pd.DataFrame) and 'ADX_14' in ax.columns: dt['adx']=ax['ADX_14']
            else: dt['adx']=ax.iloc[:,0] if len(ax.shape)>1 else ax
        else: return pd.Series(index=df.index,dtype=int)
    except Exception as e: _log_exc('regime',e); return pd.Series(index=df.index,dtype=int)
    dt = dt.dropna(subset=['vol','adx'])
    if dt.empty: return pd.Series(index=df.index,dtype=int)
    vp33,vp67 = dt['vol'].quantile([0.33,0.67]); ap33,ap67 = dt['adx'].quantile([0.33,0.67])
    def cl(row):
        v,a = row['vol'],row['adx']
        if v<vp33 and a<ap33: return 0
        elif v>vp67 or a>ap67: return 2
        return 1
    rg = dt.apply(cl,axis=1); rs = pd.Series(index=df.index,dtype=int); rs.loc[rg.index] = rg; rs.ffill(inplace=True)
    return rs
def detectar_swing_low(df,j=10,conf=True):
    if len(df)<j: return float(df['Low'].min())
    lo = df['Low']; mr = lo.rolling(window=2*j+1,center=True).min(); sw = (lo==mr)&~mr.isna()
    if conf: sw = sw&(df['Close'].shift(-1)>lo)
    sv = lo[sw]; return float(sv.iloc[-1]) if len(sv)>0 else float(lo.min())
def detectar_swing_high(df,j=10,conf=True):
    if len(df)<j: return float(df['High'].max())
    hi = df['High']; mr = hi.rolling(window=2*j+1,center=True).max(); sw = (hi==mr)&~mr.isna()
    if conf: sw = sw&(df['Close'].shift(-1)<hi)
    sv = hi[sw]; return float(sv.iloc[-1]) if len(sv)>0 else float(hi.max())
def calcular_payoff_real(ent,alv,stp,cst,dir='COMPRA'):
    if dir=='COMPRA':
        if stp>=ent or alv<=ent: return 0.0
        risco = ent-stp; ret = alv-ent
    elif dir=='VENDA':
        if stp<=ent or alv>=ent: return 0.0
        risco = stp-ent; ret = ent-alv
    else: return 0.0
    if risco<=0: return 0.0
    return round(max(0,ret-(ent+alv)*cst)/risco,2)
def detectar_regime_volatilidade(serie,j=40):
    try:
        if serie is None or len(serie)<j: return 'BAIXA'
        ret = serie.pct_change().dropna()
        if len(ret)<20: return 'BAIXA'
        vol_at = float(ret.rolling(20).std().iloc[-1])
        vol_hist = ret.rolling(j).std().dropna()
        if vol_hist.empty: return 'BAIXA'
        return 'ALTA' if float((vol_hist<vol_at).mean())>0.7 else 'BAIXA'
    except Exception as e: _log_exc('volat',e); return 'BAIXA'
def detectar_volume_anormal(df,p=20,lim=1.5):
    if len(df)<p: return False
    try:
        vm = df['Volume'].rolling(p).mean().iloc[-1]; va = df['Volume'].iloc[-1]
        return va>=vm*lim if (pd.notna(vm) and vm>0) else False
    except Exception as e: _log_exc('vol',e); return False
def fractional_kelly(wr,pr,frac=0.25):
    if pr<=0: return 0.0
    return max(0.0,min((pr*wr-(1-wr))/pr,0.25))*frac
def calcular_fibonacci_retracao(df):
    if len(df)<50: return None
    sh = detectar_swing_high(df,10,False); sl = detectar_swing_low(df,10,False)
    if sh is None or sl is None or sh<=sl or sl<=0: return None
    diff = sh-sl
    return {'38.2%':round(sh-diff*0.382,2),'50.0%':round(sh-diff*0.5,2),'61.8%':round(sh-diff*0.618,2)}
def analisar_candle(row,ant=None):
    o,h,l,c = row['Open'],row['High'],row['Low'],row['Close']
    co = abs(c-o); rt = h-l
    if rt<=0: return {}
    ps,pi = h-max(o,c),min(o,c)-l; res = {}
    if pi>=2*co and ps<=0.3*rt and co>0: res['martelo']=True
    if ps>=2*co and pi<=0.3*rt and co>0: res['estrela_cadente']=True
    if co<=0.05*rt: res['doji']=True
    if ant is not None:
        oa,ca = ant['Open'],ant['Close']; ca_ant = abs(ca-oa)
        if co<ca_ant and h<=ant['High'] and l>=ant['Low']:
            if ca<oa and c>o: res['harami_alta']=True
            elif ca>oa and c<o: res['harami_baixa']=True
        if co>ca_ant:
            if ca<oa and c>o and o<=ca and c>=oa: res['engolfo_alta']=True
            if ca>oa and c<o and o>=ca and c<=oa: res['engolfo_baixa']=True
        if ca<oa and c>o and o>ca: res['kicker_alta']=True
        if ca>oa and c<o and o<ca: res['kicker_baixa']=True
    return res
def detectar_retangulo(df,j=20):
    if len(df)<j: return None
    hh,ll = df['High'].iloc[-j:],df['Low'].iloc[-j:]
    r,s = hh.max(),ll.min()
    if r-s<0.02*s: return None
    if np.sum(hh.values>=r*0.99)>=2 and np.sum(ll.values<=s*1.01)>=2:
        return {'tipo':'Retangulo','suporte':round(s,2),'resistencia':round(r,2),'projecao':r-s}
    return None
print("✅ Célula 2 carregada")

In [ ]:
# ================ CÉLULA 3: MÓDULOS DE ANÁLISE + SETUPS (REVISADOS) ================
SETOR_POR_TICKER = {
    'PETR4':'Petróleo','PETR3':'Petróleo','PRIO3':'Petróleo',
    'VALE3':'Mineração','GGBR4':'Siderurgia','CSNA3':'Siderurgia',
    'ITUB4':'Financeiro','BBDC4':'Financeiro','BBAS3':'Financeiro',
    'ABEV3':'Consumo','MGLU3':'Varejo','RENT3':'Varejo',
    'WEGE3':'Indústria','RADL3':'Saúde','JBSS3':'Alimentos',
}
def obter_setor(t): return SETOR_POR_TICKER.get(t.replace('.SA',''),'Outros')

# ============ MÓDULO 1: TENDÊNCIA (ESQUELETO) ============
def modulo_tendencia(df):
    try:
        adx_serie = ta.adx(df['High'], df['Low'], df['Close'], length=14)
        if adx_serie is None or adx_serie.empty: return False, 'ADX indisponível', {}
        adx_val = float(adx_serie['ADX_14'].iloc[-1])
        if adx_val < ADX_MINIMO: return False, f'ADX {adx_val:.1f} < {ADX_MINIMO}', {'adx': adx_val}
        mm20 = df['Close'].rolling(20).mean().iloc[-1]
        mm50 = df['Close'].rolling(50).mean().iloc[-1]
        mm200 = df['Close'].rolling(200).mean().iloc[-1] if len(df)>=200 else mm50
        if not (mm20 > mm50 > mm200): return False, 'Médias não alinhadas', {'mm20':mm20,'mm50':mm50,'mm200':mm200}
        return True, 'Tendência alinhada', {'adx':adx_val,'mm20':mm20,'mm50':mm50,'mm200':mm200}
    except Exception as e: return False, f'Erro: {e}', {}

# Contexto de Wyckoff (não é setup, mas define direção e fase)
def modulo_wyckoff(df_w):
    if not USAR_WYCKOFF: return True, None, {}
    if len(df_w)<50: return True, None, {}
    cl = df_w['Close'].values; hi = df_w['High'].values; lo = df_w['Low'].values; vol = df_w['Volume'].values
    jr = min(200,len(df_w)); rh = np.max(hi[-jr:]); rl = np.min(lo[-jr:])
    pa = cl[-1]; amp = (rh-rl)/rl if rl>0 else 0
    if amp<0.10: return True, None, {}
    pos = (pa-rl)/(rh-rl) if rh>rl else 0.5
    vm20 = np.mean(vol[-20:]); va = vol[-1]
    c6m = np.mean(cl[-min(130,len(df_w)):-min(104,len(df_w))]) if len(df_w)>=130 else cl[0]
    ta_ant = 'ALTA' if pa>c6m*1.05 else ('BAIXA' if pa<c6m*0.95 else 'LATERAL')
    spring,sp_f = False,0.0; utad,ut_f = False,0.0
    for i in range(len(df_w)-10,len(df_w)-1):
        if lo[i]<rl*0.98 and cl[i]>rl:
            spring = True; pen = (rl-lo[i])/rl; rec = (cl[i]-lo[i])/(hi[i]-lo[i]) if hi[i]>lo[i] else 0
            sp_f = min(1.0,(pen+rec)/2); break
        if hi[i]>rh*1.02 and cl[i]<rh:
            utad = True; pen = (hi[i]-rh)/rh; rej = (hi[i]-cl[i])/(hi[i]-lo[i]) if hi[i]>lo[i] else 0
            ut_f = min(1.0,(pen+rej)/2); break
    dir_w,fase,conf,evt = 'NEUTRO',None,0.0,''
    if spring and sp_f>0.3:
        dir_w = 'COMPRA'; fase = 'C'; conf = sp_f*(0.5+0.5*(WYCKOFF_SENSIBILIDADE/200))
        evt = f'Spring (força {sp_f:.0%})'
        if va>vm20*1.5: conf+=0.15; evt+=' + clímax de venda'
        if pos<0.3: conf+=0.10; evt+=' | fundo do range'
    elif utad and ut_f>0.3:
        dir_w = 'VENDA'; fase = 'C'; conf = ut_f*(0.5+0.5*(WYCKOFF_SENSIBILIDADE/200))
        evt = f'UTAD (força {ut_f:.0%})'
        if va>vm20*1.5: conf+=0.15; evt+=' + clímax de compra'
        if pos>0.7: conf+=0.10; evt+=' | topo do range'
    elif pos<0.25 and ta_ant=='BAIXA':
        dir_w = 'COMPRA'; fase = 'A/B'; conf = 0.25*(WYCKOFF_SENSIBILIDADE/200)
        evt = 'Possível Acumulação (fundo do range)'
    elif pos>0.75 and ta_ant=='ALTA':
        dir_w = 'VENDA'; fase = 'A/B'; conf = 0.25*(WYCKOFF_SENSIBILIDADE/200)
        evt = 'Possível Distribuição (topo do range)'
    conf = min(1.0,conf*(WYCKOFF_SENSIBILIDADE/100))
    info = {'fase': fase, 'confianca': conf, 'evento': evt, 'direcao_wyckoff': dir_w}
    ok = conf >= 0.10 and dir_w in ['COMPRA', 'VENDA']
    return ok, 'Wyckoff', info

# ============ MÓDULO 2: FORÇA (MÚSCULO) ============
def modulo_forca_rsi(df):
    if not USAR_RSI: return True, None, {}
    rsi_serie = ta.rsi(df['Close'], length=14)
    if rsi_serie is None or rsi_serie.empty: return True, None, {}
    rsi_val = float(rsi_serie.iloc[-1])
    if rsi_val >= 70: return False, f'RSI {rsi_val:.1f} (sobrecomprado)', {'rsi': rsi_val}
    return True, None, {'rsi': rsi_val}

def modulo_forca_obv(df):
    if not USAR_OBV: return True, None, {}
    try:
        obv_serie = ta.obv(df['Close'], df['Volume'])
        if obv_serie is None or len(obv_serie) < 20: return True, None, {}
        obv_atual = float(obv_serie.iloc[-1])
        obv_med = obv_serie.rolling(20).mean().iloc[-1]
        if obv_atual <= obv_med: return False, 'OBV abaixo da média', {'obv': obv_atual, 'obv_med': obv_med}
        return True, None, {'obv': obv_atual, 'obv_med': obv_med}
    except: return True, None, {}

# ============ MÓDULO 3: TEMPO (CÉLULA) ============
def modulo_tempo_candle(df, entrada):
    if not USAR_CANDLE: return True, None, {}
    ef_serie = calcular_eficiencia_candle(df)
    ef_val = float(ef_serie.iloc[-1]) if not ef_serie.empty else 0.0
    if ef_val < EFICIENCIA_MINIMA_CANDLE:
        return False, f'Eficiência {ef_val:.2f} < {EFICIENCIA_MINIMA_CANDLE}', {'eficiencia': ef_val}
    pc = analisar_candle(df.iloc[-1], df.iloc[-2] if len(df) >= 2 else None)
    ok = pc.get('martelo') or pc.get('engolfo_alta') or pc.get('kicker_alta') or pc.get('harami_alta')
    if not ok: return False, 'Sem padrão de candle', {'padroes': pc}
    # Sangue: volume deve confirmar o gatilho
    if USAR_VOLUME and float(df['Volume'].iloc[-1]) <= float(df['Volume'].iloc[-2]):
        return False, 'Volume não confirma candle', {'volume_atual': float(df['Volume'].iloc[-1]), 'volume_anterior': float(df['Volume'].iloc[-2])}
    return True, None, {'eficiencia': ef_val, 'padroes': pc}

def modulo_tempo_fibonacci(df, entrada):
    if not USAR_FIBONACCI: return True, None, {}
    fib = calcular_fibonacci_retracao(df)
    if fib is None: return True, None, {}
    ok = fib['61.8%'] <= entrada <= fib['38.2%']
    return ok, 'Fibonacci', {'38.2%': fib['38.2%'], '50.0%': fib['50.0%'], '61.8%': fib['61.8%']}

# ============ MÓDULO 4: CONFIRMAÇÃO (SANGUE) ============
def modulo_confirmacao_volume(df):
    if not USAR_VOLUME: return True, None, {}
    vol_med = df['Volume'].rolling(20).mean().iloc[-1]
    vol_atual = df['Volume'].iloc[-1]
    ok = vol_atual >= vol_med
    return ok, 'Volume', {'volume_atual': vol_atual, 'volume_medio': vol_med}

# ============ MÓDULO 5: RISCO (CÉREBRO) ============
def modulo_risco_payoff(entrada, alvo, stop, custos_pct, direcao='COMPRA'):
    p = calcular_payoff_real(entrada, alvo, stop, custos_pct, direcao)
    ok = p >= PAYOFF_MINIMO
    return ok, 'Payoff', {'payoff': p}

def modulo_risco_setor(ticker, contagem_setores):
    setor = obter_setor(ticker)
    ok = contagem_setores.get(setor, 0) < MAX_ATIVOS_POR_SETOR
    return ok, setor

# ============ DETECÇÃO DE SETUPS ============
def detectar_pivo_alta(df):
    if not USAR_PIVO_ALTA or len(df)<20: return False, {}
    highs = df['High'].values; lows = df['Low'].values
    idx = len(df)-1-np.argmin(lows[-20:][::-1])
    if idx<5 or idx>=len(df)-2: return False, {}
    fundo_ant = np.min(lows[idx-10:idx])
    topo_ant = np.max(highs[idx-15:idx])
    fundo_at = lows[idx]; preco = df['Close'].iloc[-1]
    if fundo_at > fundo_ant and preco > topo_ant:
        return True, {'fundo_anterior':fundo_ant,'topo_anterior':topo_ant,'projecao':topo_ant-fundo_ant}
    return False, {}

def detectar_pivo_baixa(df):
    if not USAR_PIVO_BAIXA or len(df)<20: return False, {}
    highs = df['High'].values; lows = df['Low'].values
    idx = len(df)-1-np.argmax(highs[-20:][::-1])
    if idx<5 or idx>=len(df)-2: return False, {}
    topo_ant = np.max(highs[idx-10:idx])
    fundo_ant = np.min(lows[idx-15:idx])
    topo_at = highs[idx]; preco = df['Close'].iloc[-1]
    if topo_at < topo_ant and preco < fundo_ant:
        return True, {'topo_anterior':topo_ant,'fundo_anterior':fundo_ant,'projecao':topo_ant-fundo_ant}
    return False, {}

def detectar_fundo_duplo(df):
    if not USAR_FUNDO_DUPLO or len(df)<40: return False, {}
    lows = df['Low'].values
    sl = [(i,lows[i]) for i in range(10,len(lows)-10) if lows[i]<=min(lows[i-10:i]) and lows[i]<=min(lows[i+1:i+11])]
    if len(sl)<2: return False, {}
    f1,f2 = sl[-2],sl[-1]
    if abs(f1[1]-f2[1])/f1[1] < 0.03:
        resist = np.max(df['High'].values[f1[0]:f2[0]])
        if df['Close'].iloc[-1] > resist:
            return True, {'fundo':round(f1[1],2),'resistencia':round(resist,2),'projecao':resist-f1[1]}
    return False, {}

def detectar_topo_duplo(df):
    if not USAR_TOPO_DUPLO or len(df)<40: return False, {}
    highs = df['High'].values
    sh = [(i,highs[i]) for i in range(10,len(highs)-10) if highs[i]>=max(highs[i-10:i]) and highs[i]>=max(highs[i+1:i+11])]
    if len(sh)<2: return False, {}
    t1,t2 = sh[-2],sh[-1]
    if abs(t1[1]-t2[1])/t1[1] < 0.03:
        suporte = np.min(df['Low'].values[t1[0]:t2[0]])
        if df['Close'].iloc[-1] < suporte:
            return True, {'topo':round(t1[1],2),'suporte':round(suporte,2),'projecao':t1[1]-suporte}
    return False, {}

def detectar_topo_triplo(df):
    if not USAR_TOPO_TRIPLO or len(df)<60: return False, {}
    highs = df['High'].values
    sh = [(i,highs[i]) for i in range(10,len(highs)-10) if highs[i]>=max(highs[i-10:i]) and highs[i]>=max(highs[i+1:i+11])]
    if len(sh)<3: return False, {}
    t1,t2,t3 = sh[-3],sh[-2],sh[-1]
    if abs(t1[1]-t2[1])/t1[1] < 0.03 and abs(t2[1]-t3[1])/t2[1] < 0.03:
        suporte = min(np.min(df['Low'].values[t1[0]:t2[0]]), np.min(df['Low'].values[t2[0]:t3[0]]))
        if df['Close'].iloc[-1] < suporte:
            return True, {'topo':round(t1[1],2),'suporte':round(suporte,2),'projecao':t1[1]-suporte}
    return False, {}

def detectar_fundo_triplo(df):
    if not USAR_FUNDO_TRIPLO or len(df)<60: return False, {}
    lows = df['Low'].values
    sl = [(i,lows[i]) for i in range(10,len(lows)-10) if lows[i]<=min(lows[i-10:i]) and lows[i]<=min(lows[i+1:i+11])]
    if len(sl)<3: return False, {}
    f1,f2,f3 = sl[-3],sl[-2],sl[-1]
    if abs(f1[1]-f2[1])/f1[1] < 0.03 and abs(f2[1]-f3[1])/f2[1] < 0.03:
        resist = max(np.max(df['High'].values[f1[0]:f2[0]]), np.max(df['High'].values[f2[0]:f3[0]]))
        if df['Close'].iloc[-1] > resist:
            return True, {'fundo':round(f1[1],2),'resistencia':round(resist,2),'projecao':resist-f1[1]}
    return False, {}

def detectar_oco(df):
    if not USAR_OCO or len(df)<60: return False, {}
    highs = df['High'].values
    sh = [(i,highs[i]) for i in range(10,len(highs)-10) if highs[i]>=max(highs[i-10:i]) and highs[i]>=max(highs[i+1:i+11])]
    if len(sh)<3: return False, {}
    t1,t2,t3 = sh[-3],sh[-2],sh[-1]
    if t2[1]>t1[1] and t2[1]>t3[1] and abs(t1[1]-t3[1])/t1[1]<0.05:
        neck = np.min([np.min(df['Low'].values[t1[0]:t2[0]]),np.min(df['Low'].values[t2[0]:t3[0]])])
        if df['Close'].iloc[-1] < neck:
            return True, {'cabeca':round(t2[1],2),'neckline':round(neck,2),'projecao':t2[1]-neck}
    return False, {}

def detectar_oco_invertido(df):
    if not USAR_OCO_INVERTIDO or len(df)<60: return False, {}
    lows = df['Low'].values
    sl = [(i,lows[i]) for i in range(10,len(lows)-10) if lows[i]<=min(lows[i-10:i]) and lows[i]<=min(lows[i+1:i+11])]
    if len(sl)<3: return False, {}
    f1,f2,f3 = sl[-3],sl[-2],sl[-1]
    if f2[1]<f1[1] and f2[1]<f3[1] and abs(f1[1]-f3[1])/f1[1]<0.05:
        neck = np.max([np.max(df['High'].values[f1[0]:f2[0]]),np.max(df['High'].values[f2[0]:f3[0]])])
        if df['Close'].iloc[-1] > neck:
            return True, {'cabeca':round(f2[1],2),'neckline':round(neck,2),'projecao':neck-f2[1]}
    return False, {}

def detectar_triangulo(df):
    if not USAR_TRIANGULO or len(df)<30: return False, {}
    hh = df['High'].values[-30:]; ll = df['Low'].values[-30:]
    sh_h = [hh[i] for i in range(5,len(hh)-5) if hh[i]>=max(hh[i-5:i]) and hh[i]>=max(hh[i+1:i+6])]
    sh_l = [ll[i] for i in range(5,len(ll)-5) if ll[i]<=min(ll[i-5:i]) and ll[i]<=min(ll[i+1:i+6])]
    if len(sh_h)>=2 and len(sh_l)>=2 and sh_h[-1]<sh_h[0] and sh_l[-1]>sh_l[0]:
        return True, {'tipo':'Triângulo Simétrico','projecao':sh_h[0]-sh_l[0]}
    return False, {}

def detectar_bandeira(df):
    if not USAR_BANDEIRA or len(df)<15: return False, {}
    cls = df['Close'].values[-15:]; hh = df['High'].values[-15:]; ll = df['Low'].values[-15:]
    if cls[-1] > cls[-15]*1.05:
        mastro = cls[-1]-cls[-15]
        if mastro>0 and (max(hh[-7:])-min(ll[-7:])) < mastro*0.5:
            return True, {'tipo':'Bandeira de Alta','mastro':mastro,'projecao':mastro*2}
    return False, {}

def detectar_3_soldados(df):
    if not USAR_3_SOLDADOS or len(df)<10: return False, {}
    c1,c2,c3 = df.iloc[-3],df.iloc[-2],df.iloc[-1]
    if not (c3['Close']>c3['Open'] and c2['Close']>c2['Open'] and c1['Close']>c1['Open']): return False, {}
    co1 = abs(c1['Close']-c1['Open']); co2 = abs(c2['Close']-c2['Open']); co3 = abs(c3['Close']-c3['Open'])
    r1 = c1['High']-c1['Low'] or 0.001; r2 = c2['High']-c2['Low'] or 0.001; r3 = c3['High']-c3['Low'] or 0.001
    if co1/r1<0.5 or co2/r2<0.5 or co3/r3<0.5: return False, {}
    if not (c2['Open']>c1['Open'] and c2['Open']<c1['Close']): return False, {}
    if not (c3['Open']>c2['Open'] and c3['Open']<c2['Close']): return False, {}
    if not (c2['Close']>c1['High'] and c3['Close']>c2['High']): return False, {}
    if len(df)>=8 and df['Close'].values[-6]<df['Close'].values[-3]:
        return True, {'tipo':'3 Soldados Brancos','projecao':abs(c3['Close']-c1['Open'])}
    return False, {}

def detectar_3_corvos(df):
    if not USAR_3_CORVOS or len(df)<10: return False, {}
    c1,c2,c3 = df.iloc[-3],df.iloc[-2],df.iloc[-1]
    if not (c3['Close']<c3['Open'] and c2['Close']<c2['Open'] and c1['Close']<c1['Open']): return False, {}
    co1 = abs(c1['Close']-c1['Open']); co2 = abs(c2['Close']-c2['Open']); co3 = abs(c3['Close']-c3['Open'])
    r1 = c1['High']-c1['Low'] or 0.001; r2 = c2['High']-c2['Low'] or 0.001; r3 = c3['High']-c3['Low'] or 0.001
    if co1/r1<0.5 or co2/r2<0.5 or co3/r3<0.5: return False, {}
    if not (c2['Open']<c1['Open'] and c2['Open']>c1['Close']): return False, {}
    if not (c3['Open']<c2['Open'] and c3['Open']>c2['Close']): return False, {}
    if not (c2['Close']<c1['Low'] and c3['Close']<c2['Low']): return False, {}
    if len(df)>=8 and df['Close'].values[-6]>df['Close'].values[-3]:
        return True, {'tipo':'3 Corvos Pretos','projecao':abs(c1['Open']-c3['Close'])}
    return False, {}

def detectar_diamante(df):
    if not USAR_DIAMANTE or len(df)<40: return False, {}
    hh = df['High'].values[-40:]; ll = df['Low'].values[-40:]
    h1,l1 = hh[:20],ll[:20]; h2,l2 = hh[20:],ll[20:]
    mid = 10
    if np.max(h1)<=h1[mid] or np.min(l1)>=l1[mid]: return False, {}
    sh_d = [h2[i] for i in range(3,len(h2)-3) if h2[i]>=max(h2[i-3:i+4])]
    sl_d = [l2[i] for i in range(3,len(l2)-3) if l2[i]<=min(l2[i-3:i+4])]
    if len(sh_d)>=2 and len(sl_d)>=2 and sh_d[-1]<sh_d[0] and sl_d[-1]>sl_d[0]:
        return True, {'tipo':'Diamante','projecao':np.max(h1)-np.min(l1)}
    return False, {}

def detectar_pullback(df):
    if not USAR_PULLBACK or len(df)<10: return False, {}
    cls = df['Close'].values; hh = df['High'].values; ll = df['Low'].values
    resist = np.max(hh[-20:-5]) if len(df)>=20 else np.max(hh[:-5])
    if cls[-5] > resist and cls[-1] < cls[-5] and min(ll[-3:]) <= resist*1.02:
        return True, {'tipo':'Pullback','suporte_rompido':round(resist,2),'projecao':abs(cls[-1]-resist)}
    return False, {}

# ============ PROJEÇÃO DE ALVOS ============
def calcular_alvo_projecao(setup, info, entrada, direcao):
    proj = info.get('projecao', 0)
    if direcao == 'COMPRA':
        if setup in ['Pivô de Alta','Fundo Duplo','Fundo Triplo','Triângulo','Bandeira','Retângulo','OCO Invertido','3 Soldados Brancos']:
            if proj > 0: return entrada + proj
    elif direcao == 'VENDA':
        if setup in ['Pivô de Baixa','Topo Duplo','Topo Triplo','OCO','3 Corvos Pretos','Diamante']:
            if proj > 0: return entrada - proj
    return None

# ============ NH-NL: SAÚDE DO MERCADO ============
def calcular_nh_nl(data_w_dict):
    novas_max, novas_min, total = 0, 0, 0
    for t, df in data_w_dict.items():
        if df is None or df.empty or len(df) < 52: continue
        try:
            max_52 = df['High'].rolling(52).max().iloc[-1]
            min_52 = df['Low'].rolling(52).min().iloc[-1]
            close = df['Close'].iloc[-1]
            if pd.notna(max_52) and pd.notna(min_52) and pd.notna(close):
                if close >= max_52 * 0.995: novas_max += 1
                if close <= min_52 * 1.005: novas_min += 1
            total += 1
        except: pass
    return {'novas_max': novas_max, 'novas_min': novas_min, 'nh_nl': novas_max - novas_min,
            'total': total, 'pct_max': round(novas_max/total*100,1) if total>0 else 0,
            'pct_min': round(novas_min/total*100,1) if total>0 else 0}

def _normalizar_dataframe(df):
    df = df.copy()
    if isinstance(df.columns,pd.MultiIndex): df.columns = ['_'.join(col).strip() for col in df.columns.values]
    rm = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
    for l,o in {str(c).lower():c for c in df.columns}.items():
        if l in rm: df.rename(columns={o:rm[l]},inplace=True)
    df.sort_index(inplace=True)
    if not isinstance(df.index,pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    return df
def _safe_atr(h,l,c,ln):
    try:
        a = ta.atr(h,l,c,length=ln)
        if a is None or a.empty: return 0.0
        v = a.iloc[-1]; return float(v) if pd.notna(v) else 0.0
    except Exception as e: _log_exc('atr',e); return 0.0
print("✅ Célula 3 carregada")

In [ ]:
# ================ CÉLULA 4: EXECUÇÃO PRINCIPAL ================
def obter_tickers_b3():
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE) as f: cache = json.load(f)
            if (datetime.now()-datetime.fromisoformat(cache['timestamp'])).total_seconds()/3600<24:
                logger.log(f"📦 Cache tickers ({len(cache['tickers'])} ativos)")
                return cache['tickers']
        except: pass
    try:
        resp = requests.get("https://www.dadosdemercado.com.br/acoes",timeout=10,headers={'User-Agent':'Mozilla/5.0'})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content,'html.parser')
        tickers = [cells[0].text.strip().replace('.SA','') for row in soup.select('table tbody tr') if (cells:=row.find_all('td')) and not cells[0].text.strip().startswith('#') and cells[0].text.strip()]
        if tickers:
            with open(CACHE_TICKERS_FILE,'w') as f: json.dump({'timestamp':datetime.now().isoformat(),'tickers':tickers},f)
            logger.log(f"🌐 Scraping ({len(tickers)} ativos)")
            return tickers
    except Exception as e: logger.log(f"⚠️ Scraping: {str(e)[:80]}","WARN")
    return FALLBACK_TICKERS.copy()

def extrair_dataframe_ticker(data_raw,ticker):
    try:
        if isinstance(data_raw.columns,pd.MultiIndex):
            if ticker not in data_raw.columns.get_level_values(1) and ticker not in data_raw.columns.get_level_values(0): return None
            df = data_raw[ticker].copy()
        else: df = data_raw.copy()
        if isinstance(df.columns,pd.MultiIndex): df.columns = ['_'.join(col).strip() for col in df.columns.values]
        df.columns = [c.lower() for c in df.columns]
        rm = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
        df.rename(columns={k:rm.get(k,k) for k in df.columns if k in rm},inplace=True)
        return df
    except Exception as e: _log_exc(f'extrair {ticker}',e); return None

def resample_tf(df,freq,md=4,mdm=10):
    if df is None or df.empty: return None
    df = df.copy()
    if not isinstance(df.index,pd.DatetimeIndex):
        try: df.index = pd.to_datetime(df.index)
        except: return None
    hoje = datetime.now()
    if freq.startswith('W') and (hoje.weekday()<4 or (hoje.weekday()==4 and hoje.hour<18)):
        us = df.index[df.index.dayofweek==4]
        if len(us)>0: df = df.loc[:us[-1]]
        if df.empty: return None
    agg = {'Open':'first','High':'max','Low':'min','Close':'last','Volume':'sum'}
    dfr = df.resample(freq,closed='right',label='right').agg(agg)
    cnt = df.resample(freq,closed='right',label='right').count()['Close']
    dfr = dfr[cnt>=(md if freq.startswith('W') else mdm)]
    dfr = dfr.replace([np.inf,-np.inf],np.nan).dropna()
    return dfr[dfr['Close']>0]

# ---- FLUXO ----
logger.inicio("Coleta de Tickers")
tickers_b3 = obter_tickers_b3()
tickers_b3 = [t.replace('.SA','') for t in tickers_b3]
logger.fim("Coleta de Tickers",{'total':len(tickers_b3)})

tickers_yahoo = [t+".SA" for t in tickers_b3]
tickers_liquidos = []
BATCH = 50

logger.inicio("Filtro de Liquidez")
for i in range(0,len(tickers_yahoo),BATCH):
    batch = tickers_yahoo[i:i+BATCH]
    try:
        dr = yf.download(batch,period='3mo',interval='1d',group_by='ticker',progress=False,auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS: continue
            df = extrair_dataframe_ticker(dr,t)
            if df is None or df.empty or 'Volume' not in df.columns: continue
            try:
                vm = df['Volume'].rolling(21).mean().iloc[-1]
                pc = df['Close'].iloc[-1]
                if pd.isna(vm) or pd.isna(pc) or pc<=0: continue
                if vm>=VOLUME_MINIMO_ACAO and (vm*pc)>=VOLUME_FINANCEIRO_MINIMO: tickers_liquidos.append(t)
            except Exception as e: _log_exc(f'liq {t}',e)
    except Exception as e: logger.log(f"Erro lote {i//BATCH}: {str(e)[:100]}","ERRO")
    time.sleep(1)
if len(tickers_liquidos)<10:
    logger.log("Fallback tickers","WARN")
    tickers_liquidos = [t+".SA" for t in FALLBACK_TICKERS[:20]]
logger.fim("Filtro de Liquidez",{'liquidos':len(tickers_liquidos)})

logger.inicio("Download Histórico")
data_d = {}
for i in range(0,len(tickers_liquidos),BATCH):
    batch = tickers_liquidos[i:i+BATCH]
    try:
        dr = yf.download(batch,period='5y',interval='1d',group_by='ticker',progress=False,auto_adjust=True)
        for t in batch:
            df = extrair_dataframe_ticker(dr,t)
            if df is not None and not df.empty: data_d[t] = df
    except Exception as e: _log_exc(f'download',e)
    time.sleep(1)
logger.fim("Download Histórico",{'sucesso':len(data_d)})

logger.inicio("Resample")
data_w,data_m = {},{}
for t in tickers_liquidos:
    try:
        if t in data_d and not data_d[t].empty:
            dfd = data_d[t].copy()
            data_w[t] = resample_tf(dfd,'W-FRI')
            if data_w[t] is not None: data_m[t] = resample_tf(dfd,'ME',mdm=10)
    except Exception as e: _log_exc(f'resample {t}',e)
logger.fim("Resample",{'semanais':len(data_w)})

# ============ NH-NL DETALHADO ============
info_nhnl = calcular_nh_nl(data_w)
logger.log(f"📊 NH‑NL: {info_nhnl['nh_nl']} | Máx: {info_nhnl['novas_max']} | Mín: {info_nhnl['novas_min']}")
logger.log(f"   % Máximas: {info_nhnl['pct_max']}% | % Mínimas: {info_nhnl['pct_min']}% | Total: {info_nhnl['total']}")
if info_nhnl['nh_nl'] > 20: status_nhnl = '🟢 SAUDÁVEL'
elif info_nhnl['nh_nl'] > 0: status_nhnl = '🟡 MODERADO'
elif info_nhnl['nh_nl'] > -20: status_nhnl = '🟠 ALERTA'
else: status_nhnl = '🔴 CRÍTICO'
logger.log(f"   Diagnóstico: {status_nhnl}")

logger.inicio("Regime")
regime_vol = 'BAIXA'
try:
    for sim in ["^BVSP","^IBOV","BOVA11.SA"]:
        try:
            ib = yf.download(sim,period='3mo',interval='1d',progress=False,auto_adjust=True)
            if not ib.empty and 'Close' in ib.columns:
                ibov = ib['Close'].dropna()
                if len(ibov)>=60: break
        except: pass
    if ibov is not None and len(ibov)>=60: regime_vol = detectar_regime_volatilidade(ibov)
except Exception as e: logger.warn(f"Regime: {e}")
kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO,PAYOFF_ESTIMADO,KELLY_FRAC)

oportunidades_swing,status_ativos = [],[]
contagem_setores = {}
analisados = 0

logger.inicio("Análise de Setups")
for i,ticker in enumerate(tickers_liquidos):
    if i%20==0: logger.log(f"Progresso: {i+1}/{len(tickers_liquidos)}","DEBUG")
    df_w = data_w.get(ticker)
    if df_w is None or df_w.empty:
        status_ativos.append({'Ticker':ticker,'Direcao':'N/A','Status':'❌ Recusado','Filtro':'Sem dados','Detalhe':'Sem dados semanais'})
        continue
    analisados += 1
    dfn = _normalizar_dataframe(df_w)
    entrada = float(dfn['Close'].iloc[-1])
    if pd.isna(entrada) or entrada<=0 or entrada<PRECO_MINIMO:
        status_ativos.append({'Ticker':ticker,'Direcao':'COMPRA','Status':'❌ Recusado','Filtro':'Preço','Detalhe':f'R$ {entrada:.2f} < R$ {PRECO_MINIMO:.2f}'})
        continue
    
    # MÓDULO 1: TENDÊNCIA
    ok_tend, lbl_tend, info_tend = modulo_tendencia(dfn)
    if not ok_tend:
        status_ativos.append({'Ticker':ticker,'Direcao':'COMPRA','Status':'❌ Recusado','Filtro':'Tendência','Detalhe':lbl_tend}); continue
    ok_w, lbl_w, info_w = modulo_wyckoff(dfn)
    if not ok_w:
        status_ativos.append({'Ticker':ticker,'Direcao':info_w.get('direcao_wyckoff','COMPRA'),'Status':'❌ Recusado','Filtro':'Wyckoff','Detalhe':info_w.get('evento','?')}); continue
    direcao = info_w.get('direcao_wyckoff','COMPRA')
    
    # MÓDULO 2: FORÇA
    ok_rsi, lb_rsi, inf_rsi = modulo_forca_rsi(dfn)
    if not ok_rsi: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'RSI','Detalhe':lb_rsi}); continue
    ok_obv, lb_obv, inf_obv = modulo_forca_obv(dfn)
    if not ok_obv: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'OBV','Detalhe':lb_obv}); continue
    
    # MÓDULO 3: TEMPO
    ok_c, lb_c, inf_c = modulo_tempo_candle(dfn, entrada)
    if not ok_c: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Candle','Detalhe':lb_c}); continue
    ok_fib, lb_fib, inf_fib = modulo_tempo_fibonacci(dfn, entrada)
    if not ok_fib: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Fibonacci','Detalhe':f"Fora {inf_fib.get('61.8%',0):.2f}-{inf_fib.get('38.2%',0):.2f}"}); continue
    
    # MÓDULO 4: CONFIRMAÇÃO
    ok_v, lb_v, inf_v = modulo_confirmacao_volume(dfn)
    if not ok_v: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Volume','Detalhe':f"Vol {inf_v.get('volume_atual',0):.0f} < Med {inf_v.get('volume_medio',0):.0f}"}); continue
    
    # Stop Loss
    rl = float(dfn['Low'].rolling(window=min(52,len(dfn))).min().iloc[-1])
    atr = _safe_atr(dfn['High'],dfn['Low'],dfn['Close'],14) or entrada*0.02
    stop_atr = entrada-1.8*atr if direcao=='COMPRA' else entrada+1.8*atr
    swing_val = detectar_swing_low(dfn,12)
    sc = [s for s in [stop_atr, swing_val] if s is not None and s>0 and s<entrada]
    if not sc: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Stop','Detalhe':'Sem stop válido'}); continue
    stop_f = max(sc) if direcao=='COMPRA' else min(sc)
    risco = abs(entrada-stop_f)
    if risco/entrada<RISCO_PERCENTUAL_MINIMO or risco/entrada>RISCO_PERCENTUAL_MAXIMO:
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Risco','Detalhe':f'Risco {risco/entrada*100:.1f}%'}); continue
    
    # DETECÇÃO DE SETUP
    setup_nome = None; info_setup = {}
    if direcao == 'COMPRA':
        for fn, nome in [(detectar_pivo_alta,'Pivô de Alta'),(detectar_fundo_duplo,'Fundo Duplo'),(detectar_fundo_triplo,'Fundo Triplo'),(detectar_oco_invertido,'OCO Invertido'),(detectar_3_soldados,'3 Soldados Brancos'),(detectar_triangulo,'Triângulo'),(detectar_bandeira,'Bandeira')]:
            ok, inf = fn(dfn)
            if ok: setup_nome = nome; info_setup = inf; break
        if setup_nome is None:
            ret = detectar_retangulo(dfn)
            if ret: setup_nome = 'Retângulo'; info_setup = ret
    else:
        for fn, nome in [(detectar_pivo_baixa,'Pivô de Baixa'),(detectar_topo_duplo,'Topo Duplo'),(detectar_topo_triplo,'Topo Triplo'),(detectar_oco,'OCO'),(detectar_3_corvos,'3 Corvos Pretos')]:
            ok, inf = fn(dfn)
            if ok: setup_nome = nome; info_setup = inf; break
    if setup_nome is None:
        ok, inf = detectar_diamante(dfn)
        if ok: setup_nome = 'Diamante'; info_setup = inf
    if setup_nome is None:
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Setup','Detalhe':'Nenhum padrão detectado'}); continue
    
    # Projeção de Alvo
    alvo_proj = calcular_alvo_projecao(setup_nome, info_setup, entrada, direcao)
    alvo = alvo_proj if alvo_proj is not None else (entrada+risco*3 if direcao=='COMPRA' else entrada-risco*3)
    
    # MÓDULO 5: RISCO
    ok_pay, lb_pay, inf_pay = modulo_risco_payoff(entrada, alvo, stop_f, 0.003, direcao)
    if not ok_pay: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Payoff','Detalhe':f"Payoff {inf_pay.get('payoff',0):.2f}"}); continue
    ok_set, setor = modulo_risco_setor(ticker, contagem_setores)
    if not ok_set: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':f'Setor ({setor})','Detalhe':f'Limite de {MAX_ATIVOS_POR_SETOR}'}); continue
    
    # APROVADO
    contagem_setores[setor] = contagem_setores.get(setor,0)+1
    score = 50 + min(50, (info_tend.get('adx',0)-ADX_MINIMO)*2 + inf_c.get('eficiencia',0)*20 + inf_pay.get('payoff',0)*5)
    inst = f'Setup: {setup_nome}. Se preço atingir R$ {entrada:.2f}, {direcao} com stop R$ {stop_f:.2f} e alvo R$ {alvo:.2f}. Payoff {inf_pay.get("payoff",0):.2f}:1.'
    setup = {'Ticker':ticker,'Setup':setup_nome,'Direcao':direcao,'Entrada':round(entrada,2),'Stop Loss':round(stop_f,2),'Alvo':round(alvo,2),'Payoff':inf_pay.get('payoff',0),'Score':round(score,1),'Instrucao':inst}
    oportunidades_swing.append(setup)
    status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'✅ APROVADO','Filtro':'Nenhum','Detalhe':inst})

logger.fim("Análise de Setups",{'analisados':analisados,'aprovados':len(oportunidades_swing)})
if oportunidades_swing: oportunidades_swing = sorted(oportunidades_swing,key=lambda x:x.get('Score',0),reverse=True)[:MAX_SETUPS_POR_DIA]
logger.log(f"🎯 Oportunidades: {len(oportunidades_swing)} | Kelly: {kelly_pct*100:.2f}% | Regime: {regime_vol}","RESULTADO")

# Relatório
def gerar_relatorio(sa,op,rv,nhnl_info,kp,st_nhnl):
    ls = []
    ls.append("="*80)
    ls.append("RELATÓRIO GEBRA v12 – SETUPS DE ESPECULAÇÃO")
    ls.append(f"Data: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
    ls.append(f"Regime: {rv} | Kelly: {kp*100:.2f}%")
    ls.append(f"Oportunidades: {len(op)}")
    ls.append("="*80)
    ls.append("")
    ls.append("━━━━━━━━━━ 📊 INDICADOR NH-NL (SAÚDE DO MERCADO) ━━━━━━━━━━")
    ls.append(f"  Novas Máximas (52 sem): {nhnl_info['novas_max']} ({nhnl_info['pct_max']}%)")
    ls.append(f"  Novas Mínimas (52 sem): {nhnl_info['novas_min']} ({nhnl_info['pct_min']}%)")
    ls.append(f"  Saldo NH-NL: {nhnl_info['nh_nl']}  |  Total de ativos: {nhnl_info['total']}")
    ls.append(f"  Diagnóstico: {st_nhnl}")
    ls.append("")
    ls.append("  O que isso significa:")
    ls.append("  • Positivo = maioria das ações está em alta (tendência saudável)")
    ls.append("  • Negativo = maioria das ações está em baixa (pânico/aversão ao risco)")
    ls.append("  • Divergência NH-NL vs Preço = ALERTA VERMELHO de reversão")
    ls.append("  • Extremos (>+40 ou <-40) = clímax, possível reversão iminente")
    ls.append("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    ls.append("")
    ap = [s for s in sa if 'APROVADO' in str(s.get('Status',''))]
    rc = [s for s in sa if 'Recusado' in str(s.get('Status',''))]
    if ap:
        ls.append(f"✅ APROVADOS ({len(ap)}):")
        for s in ap:
            ls.append(f"  {s['Ticker']} | {s.get('Direcao','N/A')}")
            ls.append(f"  📌 {s.get('Detalhe','')}")
            ls.append("")
    if rc:
        ls.append(f"❌ RECUSADOS ({len(rc)}):")
        mt = Counter([s.get('Filtro','?') for s in rc])
        for m,q in mt.most_common():
            ls.append(f"    • {m}: {q}")
        ls.append("")
        for s in rc[:50]:
            ls.append(f"  {s['Ticker']} | {s.get('Direcao','N/A')} | {s.get('Filtro','?')}")
            if s.get('Detalhe'): ls.append(f"  📝 {s.get('Detalhe','')}")
            ls.append("")
        if len(rc) > 50: ls.append(f"  ... + {len(rc)-50} recusados (ver log completo)")
    ls.append("--- FIM ---")
    return "\n".join(ls)

rel = gerar_relatorio(status_ativos,oportunidades_swing,regime_vol,info_nhnl,kelly_pct,status_nhnl)
with open('relatorio_detalhado.txt','w') as f: f.write(rel)

def enviar_email(ass,c):
    if not EMAIL_REMETENTE or not SENHA_APP: logger.warn("E-mail não configurado"); return
    try:
        msg = MIMEMultipart(); msg['From']=EMAIL_REMETENTE; msg['To']=EMAIL_REMETENTE; msg['Subject']=ass
        msg.attach(MIMEText(c,'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com',465) as srv: srv.login(EMAIL_REMETENTE,SENHA_APP); srv.send_message(msg)
        logger.log("📧 E-mail enviado")
    except Exception as e: _log_exc('Email',e)

enviar_email(f"📊 GEBRA v12 - {len(oportunidades_swing)} Ops - {datetime.now().strftime('%d/%m %H:%M')}",rel)

def fmt_tg(op,rv,kp):
    if not op: return f"📊 GEBRA v12\nNenhuma oportunidade.\nRegime: {rv} | Kelly: {kp*100:.1f}%"
    m = f"🚀 GEBRA v12 - {datetime.now().strftime('%d/%m %H:%M')}\nRegime: {rv} | Kelly: {kp*100:.1f}%\n\n"
    for i,o in enumerate(op[:MAX_SETUPS_POR_DIA],1):
        m += f"{i}. <b>{o['Ticker']}</b> | {o.get('Setup','Setup')} | {o['Direcao']}\n"
        m += f"   Entrada: R$ {o['Entrada']:.2f} | Stop: R$ {o['Stop Loss']:.2f}\n"
        m += f"   Alvo: R$ {o.get('Alvo',0):.2f} | Payoff: {o.get('Payoff',0):.2f}\n"
        m += f"   Score: {o.get('Score',0)}\n   📌 {o.get('Instrucao','')}\n\n"
    return m

enviar_telegram(fmt_tg(oportunidades_swing,regime_vol,kelly_pct))

gc.collect()
logger.resumo()
print("\n✅ Concluído. Relatório: relatorio_detalhado.txt")
print(f"   NH-NL: {info_nhnl['nh_nl']} | {status_nhnl}")